# FinSync · Notebook 01 — Gmail: Fetch & Explore

Step-by-step: connect to Gmail (**read-only**), pull candidate transaction emails
for **HBL / ABL / NayaPay**, and lay them out in a table so we can design the
parsers from your *real* emails.

**This notebook is read-only** — it never writes to the database. It's for
exploration so we can see the actual wording of your notification emails.

Run the cells top to bottom.

## Prerequisites (one-time)

1. **Install deps** into the backend venv:
   ```bash
   cd backend && source .venv/bin/activate
   pip install -r ../notebooks/requirements.txt
   ```
2. **Authorize Gmail** (read-only, once):
   ```bash
   python ../scripts/gmail_authorize.py
   ```
   This stores a refresh token at `backend/secrets/gmail_token.json`.
3. **Select the kernel**: top-right of this notebook → pick the interpreter at
   `backend/.venv`.

If Gmail isn't authorized yet, the connect cell below will tell you.

In [7]:
# Install notebook deps into THIS kernel (should be the backend/.venv interpreter).
# Using sys.executable guarantees pip targets the running kernel, not some other pip on PATH.
import sys
from pathlib import Path

_here = Path.cwd()
_req = next(
    (
        p / "requirements.txt"
        for p in (_here, _here / "notebooks", _here.parent / "notebooks")
        if (p / "requirements.txt").exists()
    ),
    None,
)
if _req is None:
    raise FileNotFoundError(f"Could not find notebooks/requirements.txt starting from {_here}")

!{sys.executable} -m pip install -q -r "{_req}"
print("Installed into:", sys.executable)

Installed into: /Users/danish/Python/FinSync/backend/.venv/bin/python


In [8]:
# Make the backend `app` package importable from this notebook.
import sys
from pathlib import Path


def _find_backend(start: Path) -> Path:
    # Walk up until we find a folder containing backend/app.
    for base in (start, *start.parents):
        if (base / "backend" / "app").is_dir():
            return base / "backend"
    raise FileNotFoundError(f"Could not find the 'backend' folder above {start}")


BACKEND = _find_backend(Path.cwd())
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import pandas as pd

pd.set_option("display.max_colwidth", 140)
print("backend on path:", BACKEND)
print("kernel python:  ", sys.executable)

backend on path: /Users/danish/Python/FinSync/backend
kernel python:   /Users/danish/Python/FinSync/backend/.venv/bin/python


In [9]:
# Build an authenticated, read-only Gmail service from the stored token.
from app.services.gmail.client import build_service

service = build_service()
if service is None:
    raise RuntimeError(
        "Gmail not authorized. Run this once from the backend venv:\n"
        "    python ../scripts/gmail_authorize.py"
    )

profile = service.users().getProfile(userId="me").execute()
print("Connected as:", profile.get("emailAddress"))
print("Total messages in mailbox:", profile.get("messagesTotal"))

Connected as: test.user@example.com
Total messages in mailbox: 13023


## 1. Choose what to search

Two ways to find your transaction emails:

- **By configured senders** — uses `email_senders` from `backend/providers.yaml`.
  (Right now those are `REPLACE_WITH_...` placeholders, so this returns nothing
  until you fill them in.)
- **Manual query** — set `MANUAL_QUERY` to any
  [Gmail search](https://support.google.com/mail/answer/7190) string to explore
  before you know the exact senders, e.g.
  `"from:hbl OR subject:(transaction OR transfer)"`.

In [27]:
# --- Parameters: edit these ---
PROVIDER = "abl"            # "hbl" | "abl" | "nayapay"
START_DATE = "2025/01/01"   # Gmail after:  YYYY/MM/DD  (or None)
END_DATE = None             # Gmail before: YYYY/MM/DD  (or None)
MAX_RESULTS = 100          # cap on messages to pull; raise/lower as needed
MANUAL_QUERY = "from:myabl@abl.com"   # search a specific sender; set to None to use providers.yaml

# Show the senders currently configured per provider.
from app.config import get_providers_config

providers = get_providers_config()
for key, cfg in providers.providers.items():
    print(f"{key:8s} email_senders={cfg.email_senders}")

hbl      email_senders=['replace_with_hbl_email_sender']
abl      email_senders=['replace_with_abl_email_sender']
nayapay  email_senders=['replace_with_nayapay_email_sender']


In [28]:
# Build the query and list matching message ids.
from app.services.gmail.client import build_query, list_message_ids

if MANUAL_QUERY:
    query = MANUAL_QUERY
else:
    cfg = providers.provider(PROVIDER)
    query = build_query(cfg.email_senders, after=START_DATE, before=END_DATE)
print("Gmail query:", query)

ids = list_message_ids(service, query, max_results=MAX_RESULTS)
print(f"matched {len(ids)} messages")
if not ids:
    print(
        "\nNo matches. Either fill real senders in backend/providers.yaml, "
        "or set MANUAL_QUERY above and re-run."
    )

Gmail query: from:myabl@abl.com
matched 100 messages


## 2. Fetch and tabulate

Pull each matched message and normalize it (sender, subject, date, snippet) into
a pandas table so we can scan them quickly.

In [29]:
# Fetch each message and build a DataFrame overview.
from app.services.gmail.client import fetch_email

emails = [fetch_email(service, mid) for mid in ids]

df = pd.DataFrame(
    [
        {
            "message_id": e.message_id,
            "date": e.received_at,
            "sender": e.sender,
            "subject": e.subject,
            "snippet": (e.body or "").strip().replace("\n", " ")[:160],
        }
        for e in emails
    ]
)
df

,message_id,date,sender,subject,snippet
0,1a0d8f80f8ad6a87,2026-09-25 19:28:51+05:00,myABL@abl.com,Allied Bank Transaction Alert,"<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4.01 Transitional//EN""><html><head><title>Allied Bank Limited</title><meta http-equiv=""Content-T..."
1,1a0d6b86359f793e,2026-09-25 09:00:03+05:00,myABL@abl.com,Allied Bank Transaction Alert,"<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4.01 Transitional//EN""><html><head><title>Allied Bank Limited</title><meta http-equiv=""Content-T..."
2,1a0d294aeea03e19,2026-09-24 13:42:33+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
3,1a0b5cf5dce51c16,2026-09-18 23:37:42+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
4,1a0b5cefc4a4fd04,2026-09-18 23:37:15+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
...,...,...,...,...,...
95,19f5a52d5e8f54fe,2026-07-13 12:13:26+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
96,19f565a270e477a1,2026-07-12 17:42:53+05:00,myABL@abl.com,Allied Bank Transaction Alert,"<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4.01 Transitional//EN""><html><head><title>Allied Bank Limited</title><meta http-equiv=""Content-T..."
97,19f56525b84d643a,2026-07-12 17:34:24+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
98,19f5651fde73b09f,2026-07-12 17:34:01+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."


## 3. Inspect one full email body

This is how we design the parsers: read the real structure of a transaction
email (amount, direction, reference, date format). Change `ROW` to look at a
different message from the table above.

In [30]:
ROW = 0  # index into the table above

if not emails:
    print("No emails fetched — adjust the search parameters above and re-run.")
else:
    e = emails[ROW]
    print("FROM:   ", e.sender)
    print("SUBJECT:", e.subject)
    print("DATE:   ", e.received_at)
    print("MSG_ID: ", e.message_id)
    print("-" * 70)
    print((e.body or "")[:3000])

FROM:    myABL@abl.com
SUBJECT: Allied Bank Transaction Alert
DATE:    2026-09-25 19:28:51+05:00
MSG_ID:  1a0d8f80f8ad6a87
----------------------------------------------------------------------
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN"><html><head><title>Allied Bank Limited</title><meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"><style type="text/css"><!--td {	font-family: Tahoma, Verdana, Arial;	font-size: 11px;	font-weight: normal;	color: #666666;	text-decoration: none;}--></style></head><body leftmargin="10" topmargin="15" marginwidth="0" marginheight="0"><table width="640" border="0" align="center" cellpadding="0" cellspacing="1" bgcolor="#999999"> <tr> <td><table width="640" border="0" cellpadding="0" cellspacing="0" bgcolor="#FFFFFF"> <tr> <td>&nbsp;</td> </tr><tr><td height="30">&nbsp;</td> </tr> <tr> <td><table width="600" border="0" align="center" cellpadding="0" cellspacing="0"> <tr> <td><h3><strong><font color="#000066">Transaction

## 4. Save an anonymized sample (optional but recommended)

Once you can see a real transaction email, save a **redacted** copy to
`notebooks/samples/` — remove account numbers, names, and any codes, but keep the
wording/structure. We'll use these to implement the parsers and their tests.

The cell below writes the currently-inspected email to a file. **Open it and
redact it** before sharing. (The `samples/` folder is gitignored.)

In [31]:
# Dump the currently-inspected email to samples/ for redaction.
# NOTE: this writes the RAW email — open the file and remove personal data.
SAMPLES = Path.cwd() / "samples"
SAMPLES.mkdir(exist_ok=True)

if emails:
    ROW = globals().get("ROW", 0)
    e = emails[ROW]
    out = SAMPLES / f"{PROVIDER}_email_{ROW}_RAW_REDACT_ME.txt"
    out.write_text(
        f"FROM: {e.sender}\nSUBJECT: {e.subject}\nDATE: {e.received_at}\n"
        f"{'-' * 70}\n{e.body or ''}",
        encoding="utf-8",
    )
    print("Wrote:", out)
    print("→ Open it and redact account numbers / names before sharing.")
else:
    print("Nothing to save yet.")

Wrote: /Users/danish/Python/FinSync/notebooks/samples/abl_email_0_RAW_REDACT_ME.txt
→ Open it and redact account numbers / names before sharing.


## Next steps

- Repeat sections 1–3 for `PROVIDER = "abl"` and `"nayapay"` to see each format.
- Fill the real senders/subject patterns into `backend/providers.yaml`.
- Save 1–2 redacted samples per provider into `notebooks/samples/`.

Then we'll add **Notebook 02 — Parse & Normalize**: run each provider parser on
these samples, build a normalized `Transaction` table, and preview
fingerprint/dedup — before wiring it to Postgres.